# pyconfind — example walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/timodonnell/pyconfind/blob/main/examples/pyconfind_demo.ipynb)

[pyconfind](https://github.com/timodonnell/pyconfind) computes rotamer-based
side-chain **contact degrees** for protein structures — a fast, modern
reimplementation of [confind](https://grigoryanlab.org/confind/) whose output is
byte-for-byte identical to the original C++ binary.

This notebook shows how to:

1. install pyconfind and fetch the inputs it needs,
2. run an analysis through the **library API** (`pyconfind.analyze`),
3. **visualize** the results — a contact map, per-residue scores, and a 3D
   structure colored by contact degree.

It runs end-to-end on a free Colab CPU runtime.

## 1. Install

We install pyconfind with the optional `[fast]` extra (the Numba JIT backend)
plus `py3Dmol` for the 3D view. Takes ~30 s.

In [ ]:
%pip install -q "pyconfind[fast]" py3Dmol pandas matplotlib

## 2. Get a structure

We download a PDB file straight from the RCSB and parse it with **gemmi**.
`analyze()` accepts a pre-parsed `gemmi.Structure` directly, which keeps us
out of the filesystem and lets us inspect/manipulate the structure before
running. Here we use [1UBQ](https://www.rcsb.org/structure/1UBQ) (ubiquitin,
76 residues) — small enough to be fast, large enough for an interesting
contact map. Swap in any 4-letter PDB id.

In [ ]:
import urllib.request

import gemmi

PDB_ID = "1UBQ"  # try e.g. 1CRN, 3GB1, 1LYZ ...
pdb_text = urllib.request.urlopen(
    f"https://files.rcsb.org/download/{PDB_ID}.pdb"
).read().decode()
structure = gemmi.read_pdb_string(pdb_text)
print(f"downloaded {PDB_ID}: {len(structure[0])} chain(s)")

## 3. Rotamer library — no setup needed

The Dunbrack 2010 backbone-dependent library is downloaded automatically the
first time we call `analyze()` (~6 MB) and cached under
`platformdirs.user_cache_dir("pyconfind")`. The parsed library is then
memoized in `pyconfind.api.DEFAULT_ROTAMER_LIBRARY` so every subsequent call
in this process reuses it instantly.

In [ ]:
import platformdirs

print("rotamer library cache directory:", platformdirs.user_cache_dir("pyconfind"))

## 3. Run the analysis (library API)

One call does everything: parse the PDB, build and prune rotamers at every
position, and compute the pairwise contact degrees plus the per-residue
summaries (sum contact degree, *crowdedness*, *freedom*).

`analyze` uses the Numba backend automatically when it is installed; pass
`backend="python"` to force the pure-NumPy reference (identical results).

In [ ]:
import time

from pyconfind import analyze

t0 = time.perf_counter()
result = analyze(structure)  # passes the gemmi.Structure straight in
print(f"analyzed {len(result.positions)} residues in {time.perf_counter() - t0:.1f} s "
      f"(first call includes one-time library parse + Numba warm-up)")
print(f"found {len(result.report.contacts)} residue-residue contacts")

### What's in the result

`result.positions` is one record per residue; `result.report` holds the
contacts and per-residue arrays. Let's put the per-residue scores into a
DataFrame.

In [ ]:
import pandas as pd

# Per-residue scores as a pandas DataFrame.
rep = result.report
df = result.positions_dataframe()
df.head(10)

In [ ]:
# Strongest individual contacts (also available as a DataFrame).
contacts_df = result.contacts_dataframe()
contacts_df.nlargest(10, "degree")

## 4. Visualize

### Contact map

A residue x residue heatmap of contact degree. The band near the diagonal is
sequence-local packing; off-diagonal blocks are tertiary contacts (helix
packing, beta-sheet pairing, etc.).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = len(result.positions)
M = np.zeros((N, N))
for c in rep.contacts:
    M[c.pos_i, c.pos_j] = M[c.pos_j, c.pos_i] = c.degree

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(M, cmap="magma", origin="lower")
ax.set_xlabel("residue index")
ax.set_ylabel("residue index")
ax.set_title(f"{PDB_ID} contact-degree map")
fig.colorbar(im, ax=ax, label="contact degree")
plt.tight_layout()
plt.show()

### Compare to the C++ reference

The repo ships C++ `confind` contact maps for a few structures under
`tests/golden/`. We fetch the reference for this structure and plot it next to
pyconfind's result (and their difference). For `1CRN`, `1UBQ`, and `1EJG` the
two are byte-for-byte identical, so the difference map is all zeros.

In [ ]:
import urllib.error

from pyconfind import parse_confind_text

# C++ confind reference contact maps committed in the repo.
golden_url = (
    f"https://raw.githubusercontent.com/timodonnell/pyconfind/main/"
    f"tests/golden/{PDB_ID}.cont"
)
try:
    cpp_text = urllib.request.urlopen(golden_url).read().decode()
except urllib.error.HTTPError:
    cpp_text = None
    print(f"No committed C++ reference for {PDB_ID} (try 1CRN, 1UBQ, or 1EJG).")

if cpp_text is not None:
    ref = parse_confind_text(cpp_text)
    pos_id = [f"{p.position.chain},{p.position.resnum}{p.position.icode}"
              for p in result.positions]
    idx = {pid: i for i, pid in enumerate(pos_id)}
    Mref = np.zeros((N, N))
    for (pa, pb), deg in ref.contacts.items():
        if pa in idx and pb in idx:
            Mref[idx[pa], idx[pb]] = Mref[idx[pb], idx[pa]] = deg

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, mat, title in (
        (axes[0], Mref, "C++ confind (reference)"),
        (axes[1], M, "pyconfind"),
        (axes[2], np.abs(M - Mref), "|difference|"),
    ):
        im = ax.imshow(mat, cmap="magma", origin="lower")
        ax.set_title(title)
        ax.set_xlabel("residue index")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    axes[0].set_ylabel("residue index")
    fig.suptitle(f"{PDB_ID}: pyconfind vs C++ reference", fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"max |pyconfind - C++| over the contact map: {np.abs(M - Mref).max():.2e}")

### Per-residue scores

* **sum contact degree** — total contact a residue makes (buried, well-packed
  residues score high).
* **crowdedness** — fraction of rotamers pruned by backbone clashes.
* **freedom** — how much rotameric freedom the position retains.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
x = df["resnum"]
axes[0].bar(x, df["sumcond"], color="#2563eb")
axes[0].set_ylabel("sum contact\ndegree")
axes[1].bar(x, df["crwdnes"], color="#dc2626")
axes[1].set_ylabel("crowdedness")
axes[2].bar(x, df["freedom"], color="#16a34a")
axes[2].set_ylabel("freedom")
axes[2].set_xlabel("residue number")
axes[0].set_title(f"{PDB_ID} per-residue scores")
plt.tight_layout()
plt.show()

### 3D structure colored by contact degree

We write each residue's sum contact degree into the PDB B-factor column and let
[py3Dmol](https://3dmol.csb.pitt.edu/) color the cartoon by it — buried,
well-packed residues light up.

In [ ]:
import py3Dmol

score = {
    (p.position.chain, p.position.resnum): rep.sum_contact_degree[i]
    for i, p in enumerate(result.positions)
}
vmax = float(np.nanmax(rep.sum_contact_degree))

# Re-emit the downloaded PDB text with each residue's contact score in the
# B-factor column (py3Dmol colors the cartoon by it).
out_lines = []
for line in pdb_text.splitlines(keepends=True):
    if line.startswith(("ATOM", "HETATM")):
        chain = line[21].strip() or "_"
        try:
            resnum = int(line[22:26])
        except ValueError:
            out_lines.append(line)
            continue
        b = score.get((chain, resnum))
        if b is not None:
            line = f"{line[:60]}{b:6.2f}{line[66:]}"
    out_lines.append(line)
scored_pdb = "".join(out_lines)

view = py3Dmol.view(width=720, height=520)
view.addModel(scored_pdb, "pdb")
view.setStyle(
    {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0, "max": vmax}}}
)
view.zoomTo()
view.show()

## 5. Other outputs & options

pyconfind can emit the original confind text format (a drop-in replacement for
downstream tools) or structured JSON, and supports selection strings
(`focus=`/`pre_select=`) and a native-only mode.

In [ ]:
from pyconfind import format_confind_text, format_json

# Original confind text format (first lines)
print("\n".join(format_confind_text(result.positions, result.report).splitlines()[:6]))
print("...\n")

# Restrict the computed/output residues with an MSL selection string
focused = analyze(structure, focus="resi 1-20")
print(f"focused run: {len(focused.report.sum_contact_degree.nonzero()[0])} residues with contacts")

# Structured JSON for pipelines
import json
payload = json.loads(format_json(result.positions, result.report))
print("JSON keys:", list(payload))

## 6. Antibody/antigen interface — 5TRU (ipilimumab Fab + CTLA-4)

[5TRU](https://www.rcsb.org/structure/5TRU) is the immune-checkpoint inhibitor
**ipilimumab Fab** bound to **human CTLA-4** — the kind of antibody/antigen
complex you'd actually want to map. A couple of notes about this run:

* The asymmetric unit holds *two* independent Fab/CTLA-4 complexes (chains
  `L H C` and `l h c`). By default `analyze()` picks the **first biological
  assembly**, restricting analysis to a single complex; pass `assembly=None`
  to keep the whole AU, or `assembly="2"` for the other copy.
* For an existing antibody we don't want to swap amino acids — we want to
  characterize the side chains that are actually there. So we pass
  `native_only=True`, which keeps each position's native AA (still using all
  rotamers of that AA).

In [ ]:
AB_PDB_ID = "5TRU"
ab_text = urllib.request.urlopen(
    f"https://files.rcsb.org/download/{AB_PDB_ID}.pdb"
).read().decode()
ab_structure = gemmi.read_pdb_string(ab_text)

ab = analyze(ab_structure, native_only=True)   # default assembly=1 -> chains L, H, C
ab_pos = ab.positions_dataframe()
ab_contacts = ab.contacts_dataframe()

print(f"{AB_PDB_ID} (bio assembly 1): "
      f"{len(ab_pos)} residues across chains {sorted(ab_pos['chain'].unique())}, "
      f"{len(ab_contacts)} pairwise contacts")
ab_pos["chain"].value_counts().rename_axis("chain").to_frame("residues")

### Paratope-epitope contacts

The interface between the Fab (chains `L`, `H`) and CTLA-4 (chain `C`) — the
**paratope/epitope** — is captured by the cross-chain rows of
`contacts_dataframe()`. The strongest contributors are the expected CDR-H3
positions (W101, Y53, N57) packing into CTLA-4's `MYPPPY` loop.

In [ ]:
# Pull cross-chain contacts and plot the Fab x CTLA-4 interface map.
# (Normalized so the Fab side is always on the i columns.)
fab_chains = {"L", "H"}
ag_chain = "C"

ci, cj = ab_contacts["chain_i"], ab_contacts["chain_j"]
fab_i = ci.isin(fab_chains) & (cj == ag_chain)
fab_j = cj.isin(fab_chains) & (ci == ag_chain)

cols_i = ["chain_i", "resnum_i", "icode_i", "resname_i"]
cols_j = ["chain_j", "resnum_j", "icode_j", "resname_j"]
swap = ab_contacts.loc[fab_j].rename(columns=dict(zip(cols_i + cols_j, cols_j + cols_i)))
interface = pd.concat([ab_contacts.loc[fab_i], swap], ignore_index=True)[
    cols_i + cols_j + ["degree"]
]

print(f"{len(interface)} Fab <-> CTLA-4 contacts; top 10:")
display(interface.nlargest(10, "degree").reset_index(drop=True))

# Build a Fab residue x CTLA-4 residue matrix to plot.
fab_keys = [(c, n) for c, n in zip(ab_pos["chain"], ab_pos["resnum"]) if c in fab_chains]
ag_keys  = [(c, n) for c, n in zip(ab_pos["chain"], ab_pos["resnum"]) if c == ag_chain]
fab_idx  = {k: i for i, k in enumerate(fab_keys)}
ag_idx   = {k: i for i, k in enumerate(ag_keys)}

Iface = np.zeros((len(fab_keys), len(ag_keys)))
for row in interface.itertuples(index=False):
    Iface[fab_idx[(row.chain_i, row.resnum_i)], ag_idx[(row.chain_j, row.resnum_j)]] = row.degree

fig, ax = plt.subplots(figsize=(7, 9))
im = ax.imshow(Iface, cmap="magma", aspect="auto", origin="lower")
ax.set_xticks(range(0, len(ag_keys), 5))
ax.set_xticklabels([f"{c}{n}" for c, n in ag_keys[::5]], rotation=90)
h_start = next(i for i, (c, _) in enumerate(fab_keys) if c == "H")
ax.axhline(h_start - 0.5, color="white", lw=0.7, ls="--")
ax.set_yticks([h_start // 2, h_start + (len(fab_keys) - h_start) // 2])
ax.set_yticklabels(["L", "H"])
ax.set_xlabel("CTLA-4 (chain C)")
ax.set_ylabel("Fab")
ax.set_title("ipilimumab Fab <-> CTLA-4 contact interface")
fig.colorbar(im, ax=ax, label="contact degree")
plt.tight_layout()
plt.show()

---
That's it. See the [pyconfind repo](https://github.com/timodonnell/pyconfind)
for the CLI (`pyconfind --p input.pdb --rLib rotlibs --o out.cont`), the
benchmark notes, and the validation against the C++ reference.